# Chapter 8 · Long-running tasks and reliability

A client of Northside Veterinary Clinic sees two charges for the same invoice. We reproduce the failure with a payment gateway that is sometimes slower than the client is willing to wait, and we try four retry policies in the host, without a model, and four hosts with the agent: with and without idempotency keys, with and without a way to check the status of the charge.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. The first part does not need it.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the data and the gateway

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
"""Chapter 8: long-running tasks and reliability. A client sees two charges for the same
invoice: timeouts, retries and idempotency keys."""
import asyncio
import hashlib
import json
import re
import time
import sys
import unicodedata
import urllib.request
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, MCPError
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = ("https://raw.githubusercontent.com/HernanDiaz/"
              "ai-tools-mcp-skills/main/en/")
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))


def strip_accents(text):
    """Lowercase and without diacritics, to compare names however they are written."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def load_data():
    """The clinic's clients."""
    global OWNERS
    OWNERS = download("owners.json")

In [ ]:
WAIT = 1.0                        # seconds the client waits for each answer


SLOW = 1.5                        # how long the bank takes when it is slow


def load_charges():
    """The clinic's unpaid invoices, with a speed profile for their first charge."""
    global INVOICES, UNPAID, PROFILES
    INVOICES = {i["id"]: i for i in download("invoices.json")}
    UNPAID = [i for i, inv in INVOICES.items() if not inv["paid"]]
    cycle = ["fast", "charges, then slow", "fast", "slow, then charges"]
    PROFILES = {i: cycle[n % len(cycle)] for n, i in enumerate(UNPAID)}
    reset_charges()


def reset_charges():
    """CHARGES are the charges the bank has made; CONFIRMED, the invoices whose charge
    was confirmed to the clinic (the ones its billing program takes as paid)."""
    global CHARGES, CONFIRMED, ATTEMPTS, KEYS
    CHARGES, CONFIRMED, ATTEMPTS, KEYS = [], set(), {}, {}


async def bank(invoice, profile):
    """The bank: it charges the invoice's amount, before or after being slow."""
    if profile == "slow, then charges":
        await asyncio.sleep(SLOW)
    CHARGES.append(invoice)
    await asyncio.sleep(SLOW if profile == "charges, then slow" else 0.05)
    return {"charged": invoice, "amount": INVOICES[invoice]["total"]}


def create_gateway(idempotency=False, with_status=True):
    """The payments server. With idempotency, charge_invoice accepts a key: if it has
    already received that key, it does not charge again, but returns the result of the
    first charge. With with_status, it also offers invoice_status, which asks the bank
    whether the invoice has been charged."""
    server = MCPServer("payments")

    @server.tool()
    async def charge_invoice(
            invoice: Annotated[str, Field(description="Invoice id, like INV-0024.")],
            key: Annotated[str, Field(description="Idempotency key: the same in the "
                                      "retries of the same charge.")] = "") -> dict:
        """Charges an invoice to the client's card."""
        if invoice not in INVOICES:
            raise ToolError(f"Invoice {invoice} does not exist.")
        if idempotency and key in KEYS:
            result = await asyncio.shield(KEYS[key])  # the same charge, not a new one
            CONFIRMED.add(invoice)
            return result
        ATTEMPTS[invoice] = ATTEMPTS.get(invoice, 0) + 1
        profile = PROFILES.get(invoice, "fast") if ATTEMPTS[invoice] == 1 else "fast"
        order = asyncio.ensure_future(bank(invoice, profile))  # the order goes out
        if idempotency and key:
            KEYS[key] = order
        result = await asyncio.shield(order)  # waiting is not the same as charging
        CONFIRMED.add(invoice)
        return result

    if with_status:
        @server.tool()
        def invoice_status(
                invoice: Annotated[str, Field(description="Invoice id.")]) -> dict:
            """Asks the bank whether an invoice has already been charged."""
            return {"invoice": invoice, "charged": invoice in CHARGES}

    @server.tool()
    def unpaid_invoices(
            owner: Annotated[str, Field(description="Client's name.")]) -> dict:
        """Unpaid invoices of a client, with their amount."""
        ids = [c for c, x in OWNERS.items()
               if strip_accents(owner) in strip_accents(x["name"])]
        return {"invoices": [{"id": i, "total": inv["total"]}
                             for i, inv in INVOICES.items() if inv["owner"] in ids
                             and not inv["paid"] and i not in CONFIRMED]}

    return server

In [ ]:
load_data()
load_charges()
print(PROFILES)

## 2. What happens when the time runs out

The client stops waiting after one second, but the bank has already received the order.

In [ ]:
async def try_it():
    reset_charges()
    async with Client(create_gateway(), read_timeout_seconds=WAIT) as c:
        try:
            await c.call_tool("charge_invoice", {"invoice": "INV-0014"})
        except MCPError as e:
            print("The client:", e)
    await asyncio.sleep(SLOW)
    print("The bank has charged:", CHARGES)

await try_it()

## 3. Four retry policies, without a model

In [ ]:
async def charge_with(client, invoice, policy):
    """A charge with a retry policy. Returns whether the host takes it as done."""
    key = f"charge-{invoice}" if policy == "idempotency key" else ""
    for attempt in range(3):
        try:
            r = await client.call_tool("charge_invoice", {"invoice": invoice,
                                                          "key": key})
            return not r.is_error
        except MCPError:                                  # timeout
            if policy == "no retries":
                return False
            if policy == "check before retrying":
                status = await client.call_tool("invoice_status", {"invoice": invoice})
                if json.loads(status.content[0].text)["charged"]:
                    return True
    return False


POLICIES = ["no retries", "retry", "check before retrying", "idempotency key"]


async def charge_all(policy):
    """The host charges the thirteen unpaid invoices with a policy, and counts."""
    reset_charges()
    start = time.perf_counter()
    async with Client(create_gateway(idempotency=True), read_timeout_seconds=WAIT) as c:
        done = {i: await charge_with(c, i, policy) for i in UNPAID}
    await asyncio.sleep(SLOW + 0.2)                     # the orders still on their way
    times = {i: CHARGES.count(i) for i in UNPAID}
    return {"duplicate_charges": sum(max(0, n - 1) for n in times.values()),
            "charged": sum(n > 0 for n in times.values()),
            "charged_but_taken_as_failed": sum(times[i] > 0 and not done[i]
                                               for i in UNPAID),
            "seconds": round(time.perf_counter() - start, 1)}

In [ ]:
for policy in POLICIES:
    print(f"{policy:32s}", await charge_all(policy))

## 4. The agent charges

If the answer says that the invoice has not been charged, the receptionist repeats the request.

In [ ]:
tok, model = load(MODEL)

In [ ]:
SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer from the model, and its input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds and the model does not
    need."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """The JSON of a result on a single line, without the indentation the server
    sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 1's loop, with the tools of the MCP servers."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_input, n_output = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_input, "thought": thought,
                      "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                arguments = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({arguments})")
                print(f"     → {text[:150]}")
    return None, trace

In [ ]:
class Host:
    """Chapter 4's compact host, for the gateway. If a call times out, it tells the
    model with an error. With keys, it adds to each charge an idempotency key derived
    from the task and the arguments: if the model retries the same charge, the key is
    the same and the gateway does not charge twice. With explain, the timeout error
    also says what to do: not to repeat the charge and to check its status."""

    def __init__(self, server, keys=False, explain=False):
        """Keeps the server and the host's options."""
        self.server, self.keys, self.explain, self.task = server, keys, explain, ""

    async def __aenter__(self):
        """Connects and lists the tools, without the key parameter."""
        self.client = Client(self.server, read_timeout_seconds=WAIT)
        await self.client.__aenter__()
        self.schemas = []
        for t in (await self.client.list_tools()).tools:
            parameters = without_titles(t.input_schema)
            parameters["properties"].pop("key", None)        # the host puts the key
            self.schemas.append({"type": "function", "function": {
                "name": t.name, "description": t.description,
                "parameters": parameters}})
        return self

    async def __aexit__(self, *error):
        """Closes the connection."""
        await self.client.__aexit__(*error)

    async def call(self, name, arguments):
        """Runs a call of the model, with a key and a worded timeout error."""
        if name == "charge_invoice" and self.keys:
            fingerprint = json.dumps([self.task, arguments], sort_keys=True,
                                     ensure_ascii=False)
            key = hashlib.sha256(fingerprint.encode()).hexdigest()[:16]
            arguments = {**arguments, "key": key}
        try:
            r = await self.client.call_tool(name, arguments)
        except MCPError:
            error = (f"The payment gateway did not answer within {WAIT:g} s; it is not "
                     "known whether the operation was done.")
            if self.explain:
                error += (" The charge may in fact have been made: do not repeat it. "
                          "Check invoice_status to find out before answering.")
            return json.dumps({"error": error}, ensure_ascii=False)
        return compact("\n".join(b.text for b in r.content if b.type == "text"))


def charge_tasks():
    """One task for each unpaid invoice of the first six clients."""
    tasks = []
    for i in UNPAID[:6]:
        owner = OWNERS[INVOICES[i]["owner"]]["name"]
        tasks.append((f"charge_{i}", f"Charge invoice {i} to {owner}.", i))
    return tasks


def says_charged(answer):
    """Does the answer say that the invoice has been charged?"""
    if says(answer, r"already (been )?(charged|paid)",
            r"(was|is|had been|has been) already (charged|paid)"):
        return True
    return says(answer, r"\bcharged\b", r"\bpaid\b", r"\bprocessed\b",
                r"\bcompleted\b", r"went through") and not says(
        answer, r"\b(has|have|had|was|were|could|can|did)(n't| not)\b",
        r"\bcannot\b", r"\bunable\b", r"(don't|do not) know", r"\bunknown\b",
        r"not (yet )?(known|clear|confirmed)", r"\bunclear\b", r"no confirmation")


async def agent_charges(keys, explain, with_status):
    """The six charge tasks with the agent. If the answer says the invoice has not been
    charged, the receptionist repeats the same request, as anyone would. The duplicate
    charges are counted, and whether the last answer says correctly what happened."""
    rows = []
    for key, task, invoice in charge_tasks():
        reset_charges()
        answers, calls = [], []
        async with Host(create_gateway(True, with_status), keys, explain) as host:
            host.task = task
            for _ in range(2):
                answer, trace = await mcp_agent(task, host)
                answers.append(answer)
                calls += [(c["name"], c["arguments"])
                          for p in trace for c in p["calls"]]
                await asyncio.sleep(SLOW + 0.2)          # the orders still on their way
                if says_charged(answer):
                    break
        times = CHARGES.count(invoice)
        rows.append({"task": key, "profile": PROFILES[invoice], "charges": times,
                     "requests": len(answers),
                     "correct_answer": says_charged(answers[-1]) == (times > 0),
                     "answers": answers, "calls": calls})
    return {"duplicate_charges": sum(max(0, r["charges"] - 1) for r in rows),
            "repeated_requests": sum(r["requests"] - 1 for r in rows),
            "correct_answers": sum(r["correct_answer"] for r in rows), "rows": rows}


VARIANTS = {"no keys, no status check": (False, False, False),
            "keys, no status check": (True, False, False),
            "no keys, status check": (False, False, True),
            "keys and an error that explains": (True, True, True)}

In [ ]:
for name, (keys, explain, with_status) in VARIANTS.items():
    r = await agent_charges(keys, explain, with_status)
    print(f"{name:36s}", {k: v for k, v in r.items() if k != "rows"})

## Exercises

1. Explain in your own words why a timeout does not mean that the operation has failed.

2. Raise the timeout to 2 seconds and repeat the four policies. Do the duplicates disappear? What would happen if the bank took 3?

3. Change the profiles so that the gateway is always slow to charge ("slow, then charges"). How many duplicates does the policy that checks before retrying give?

4. With the idempotency key, what happens if the charges of two different invoices arrive with the same key? How should the gateway defend itself?

5. The host's key is derived from the task and the arguments. Change one word of the receptionist's repeated request. Does the key still protect?

6. Make `charge_invoice` report its progress with `ctx.report_progress` while it waits for the bank, and make the client show it with `progress_callback`. Does anything change in the duplicates?

7. Write a policy that, after a timeout, waits a little longer before checking the status. How long do you have to wait so that nothing is duplicated?

8. Add to the gateway a limit of three charges per minute and make the host retry with a growing wait. How many charges go through and how long do they take?

9. Which of the clinic's tools would need an idempotency key? Classify those of the previous chapters into queries and actions, and say in which ones a retry would do harm.

10. **Project.** Add idempotency keys to a tool that acts in your project from chapter 4, and write a server test (chapter 5) that shows that a retry with the same key does not repeat the action.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [MCP specification](https://modelcontextprotocol.io/specification): cancellation, progress and timeouts. [Python SDK](https://github.com/modelcontextprotocol/python-sdk). Northside Veterinary Clinic and its data are fictional. Original code and experiments.